# Lab 03 — Multipart Upload: Uploading Large Files Efficiently

For large objects (typically **> 100 MB**), a single `PUT` request is fragile:
a network blip means restarting from zero. The S3 **Multipart Upload API** solves this
by splitting the file into independent **parts** that are:
- Uploaded **in parallel** to saturate available bandwidth
- **Individually retried** on failure — only the failed part, not the whole file
- Assembled server-side into a single object when all parts succeed

### Multipart API lifecycle

```
CreateMultipartUpload  →  get UploadId
     UploadPart ×N     →  get ETag per part   (parallel)
     ListParts         →  verify parts
CompleteMultipartUpload →  server assembles the final object
```

If something goes wrong before `Complete`, call `AbortMultipartUpload` to discard
the uploaded parts and stop being charged for their storage.

| API limit | Value |
|-----------|-------|
| Single `PUT` max | 5 GB |
| Multipart max object size | 5 TB |
| Min part size (except last) | 5 MB |
| Max part count | 10,000 |


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`


---
## Setup — Connect and Create Bucket


In [ ]:
import boto3
import os
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab3-multipart'
OBJECT_KEY = 'uploads/large_file.bin'

# Create the bucket if it doesn't exist
existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


---
## 1 · Generate a Large Test File (~50 MB)

We generate a 50 MB binary file filled with a repeating pattern.
Using a non-zero pattern lets us verify integrity: if every byte is `0x00`,
a corrupted all-zeros file looks the same as the original.


In [ ]:
FILE_PATH = '../temp/staging/large_file.bin'
PART_SIZE = 10 * 1024 * 1024   # 10 MB per part → 5 parts for a 50 MB file
TOTAL_PARTS = 5

os.makedirs('../temp/', exist_ok=True)

# Write 50 MB with a recognizable repeating pattern
pattern = bytes(range(256)) * 4096  # 1 MB of 0x00–0xFF cycle
with open(FILE_PATH, 'wb') as f:
    for _ in range(TOTAL_PARTS * (PART_SIZE // len(pattern))):
        f.write(pattern)

file_size = os.path.getsize(FILE_PATH)
file_md5 = hashlib.md5(open(FILE_PATH, 'rb').read()).hexdigest()

print(f'✅ Test file created: {FILE_PATH}')
print(f'   Size:  {file_size / (1024*1024):.0f} MB ({file_size:,} bytes)')
print(f'   MD5:   {file_md5}  (we will verify this after reassembly)')
print(f'   Parts: {TOTAL_PARTS} × {PART_SIZE // (1024*1024)} MB each')


---
## 2 · Initiate the Multipart Upload

`create_multipart_upload` reserves the upload session and returns an **UploadId**.
This ID ties all subsequent `UploadPart` calls together. Nothing is stored in the
bucket yet — the object only appears after `CompleteMultipartUpload` succeeds.


In [ ]:
response = s3.create_multipart_upload(
    Bucket=BUCKET,
    Key=OBJECT_KEY,
    ContentType='application/octet-stream',
)
upload_id = response['UploadId']

print(f'✅ Multipart upload initiated')
print(f'   Bucket:    {BUCKET}')
print(f'   Key:       {OBJECT_KEY}')
print(f'   UploadId:  {upload_id}')
print()
print('The object does NOT exist in S3 yet — it will appear only after Complete.')


---
## 3 · Upload Parts in Parallel

Each part is uploaded independently with its own `PartNumber` (1-based).
Using `ThreadPoolExecutor` allows multiple parts to fly simultaneously,
saturating your uplink instead of uploading sequentially.

S3 returns an **ETag** for each part — a content hash we must include in
the `CompleteMultipartUpload` call to prove we have all parts.


In [ ]:
completed_parts = []   # will be sorted and passed to Complete

def upload_part(part_number):
    offset = (part_number - 1) * PART_SIZE
    with open(FILE_PATH, 'rb') as f:
        f.seek(offset)
        data = f.read(PART_SIZE)
    resp = s3.upload_part(
        Bucket=BUCKET,
        Key=OBJECT_KEY,
        UploadId=upload_id,
        PartNumber=part_number,
        Body=data,
    )
    return part_number, resp['ETag']

print(f'Uploading {TOTAL_PARTS} parts in parallel...')
with ThreadPoolExecutor(max_workers=TOTAL_PARTS) as executor:
    futures = {executor.submit(upload_part, i): i for i in range(1, TOTAL_PARTS + 1)}
    for future in as_completed(futures):
        pnum, etag = future.result()
        completed_parts.append({'PartNumber': pnum, 'ETag': etag})
        print(f'  Part {pnum} ✅  ETag: {etag}')

# Parts must be in order for the Complete call
completed_parts.sort(key=lambda p: p['PartNumber'])
print(f'\nAll {TOTAL_PARTS} parts uploaded.')


---
## 4 · Verify Uploaded Parts

`list_parts` shows all parts currently staged for a given `UploadId`.
Use this to audit what arrived before calling `Complete`,
or to resume a partial upload after a failure.


In [ ]:
response = s3.list_parts(
    Bucket=BUCKET,
    Key=OBJECT_KEY,
    UploadId=upload_id,
)

parts = response['Parts']
print(f'📦 Parts staged for upload_id={upload_id[:16]}...:')
print(f'   {"Part":>5}  {"Size (bytes)":>14}  ETag')
print('   ' + '-' * 55)
total_staged = 0
for p in parts:
    print(f'   {p["PartNumber"]:>5}  {p["Size"]:>14,}  {p["ETag"]}')
    total_staged += p['Size']
print(f'\n   Total staged: {total_staged:,} bytes ({total_staged / (1024*1024):.0f} MB)')


---
## 5 · Complete the Multipart Upload

`complete_multipart_upload` tells S3 to assemble the parts into a single object.
You must pass the `PartNumber` + `ETag` for every part in ascending order.
After this call succeeds, the object is immediately accessible in the bucket.


In [ ]:
s3.complete_multipart_upload(
    Bucket=BUCKET,
    Key=OBJECT_KEY,
    UploadId=upload_id,
    MultipartUpload={'Parts': completed_parts},
)

print(f'✅ Multipart upload complete!')
print(f'   Object is now accessible at s3://{BUCKET}/{OBJECT_KEY}')


---
## 6 · Verify the Assembled Object — Size + Content Integrity

`head_object` confirms the final object's size without downloading it.
We then **download the object and recompute its MD5**, comparing against
the pre-upload `file_md5` to guarantee byte-for-byte integrity.

> **Note:** The S3 `ETag` of a multipart-assembled object is *not* the file's MD5 —
> it's the MD5 of the concatenated part ETags followed by `-N` (the part count).
> Always download and rehash the content if you need true integrity verification.


In [ ]:
head = s3.head_object(Bucket=BUCKET, Key=OBJECT_KEY)
stored_size = head['ContentLength']
original_size = os.path.getsize(FILE_PATH)

print('🔍 Object verification:')
print(f'   Original file size: {original_size:,} bytes')
print(f'   Stored object size: {stored_size:,} bytes')

if stored_size == original_size:
    print('   ✅ Sizes match — object assembled correctly!')
else:
    raise AssertionError(f'Size mismatch: expected {original_size}, got {stored_size}')

# Download and verify MD5 for byte-for-byte integrity
print('\n🔐 Content integrity check...')
body = s3.get_object(Bucket=BUCKET, Key=OBJECT_KEY)['Body'].read()
downloaded_md5 = hashlib.md5(body).hexdigest()

print(f'   Original MD5:   {file_md5}')
print(f'   Downloaded MD5: {downloaded_md5}')

if downloaded_md5 == file_md5:
    print('   ✅ MD5 match — byte-for-byte integrity confirmed!')
else:
    raise AssertionError('MD5 mismatch — content corrupted during multipart reassembly!')

print(f'\n   ETag: {head["ETag"]}')
print("   (Note: S3 ETag for multipart ≠ file MD5. It's MD5 of part ETags + part count.)")


---
## 7 · Simulating an Upload Failure — Abort

If an upload fails mid-way (network error, application crash), you should call
`abort_multipart_upload` to clean up the partially uploaded parts.
Incomplete multipart uploads **consume storage space** until explicitly aborted.

Here we start a new upload, upload one part, then abort to demonstrate the pattern.


In [ ]:
ABORT_KEY = 'uploads/abort_demo.bin'

# Start a new upload session
abort_resp = s3.create_multipart_upload(Bucket=BUCKET, Key=ABORT_KEY)
abort_id = abort_resp['UploadId']
print(f'Started upload: UploadId={abort_id[:16]}...')

# Upload one part (simulating a partial upload)
s3.upload_part(
    Bucket=BUCKET, Key=ABORT_KEY, UploadId=abort_id,
    PartNumber=1, Body=b'x' * (5 * 1024 * 1024),  # 5 MB minimum part
)
print('Part 1 uploaded — simulating a crash before completion...')

# Abort: discards all uploaded parts and frees storage
s3.abort_multipart_upload(Bucket=BUCKET, Key=ABORT_KEY, UploadId=abort_id)
print(f'✅ Upload aborted. Parts discarded, storage freed.')
print()
print('Production tip: configure an S3 lifecycle rule to auto-abort')
print('incomplete multipart uploads after N days.')


---
## 8 · Cleanup


In [ ]:
# Delete the assembled object
s3.delete_object(Bucket=BUCKET, Key=OBJECT_KEY)
print(f'🗑️  Deleted s3://{BUCKET}/{OBJECT_KEY}')

# Remove the local test file
if os.path.exists(FILE_PATH):
    os.remove(FILE_PATH)
    print(f'🗑️  Removed local file: {FILE_PATH}')

# Delete the bucket
s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Step | API call | Purpose |
|------|----------|---------|
| 1 | `create_multipart_upload` | Reserve the session, get `UploadId` |
| 2 | `upload_part` × N | Upload each part (parallel, each ≥ 5 MB) |
| 3 | `list_parts` | Audit staged parts before completing |
| 4 | `complete_multipart_upload` | Assemble parts into the final object |
| — | `abort_multipart_upload` | Discard parts if upload fails |

### When to use multipart upload

| File size | Recommendation |
|-----------|---------------|
| < 100 MB | Single `put_object` is simpler |
| 100 MB – 1 GB | Multipart with 8–16 MB parts |
| > 1 GB | Multipart mandatory (single PUT limit is 5 GB, and reliability suffers) |

### Next steps

- **Lab 04**: Fault tolerance — how EC:2 protects data across drives
- **Lab 05**: Versioning — track multiple revisions of large uploaded files
